# Biohub cell tracking: Swin + blob detector, offline-safe, writes `submission.csv`

**What changed compared with the first version** (all of it follows from the 5,132-node result):

| Problem | Fix in this notebook |
|---|---|
| Only ~13 cells per frame found (hundreds exist) | New **blob detector** (difference of Gaussians) finds nearly every nucleus; the Swin net is trained on annotations **plus blob peaks**, so unannotated cells are no longer taught as background; `DETECTOR_MODE="hybrid"` merges both |
| Weak greedy linking | **Hungarian matching with constant-velocity prediction**, 1-frame **gap closing** |
| No divisions | Conservative **division** step (tight distance gates and caps) |
| `pip install` fails with internet off | **Offline-safe** zarr setup (uses installed zarr 3, or wheels from an attached dataset) |
| Only a recall proxy | **Local edge Jaccard** that follows the official `metrics.md` rules, on held-out training movies |
| Kernel died (out of memory) during Swin inference | **Memory leak fixed** in `detect_and_pool_streaming`: stored feature views kept every tile alive. Features are no longer collected (nothing here uses them) |
| Any crash lost the whole run | Swin training / inference are wrapped, so the run falls back to the blob detector and still writes `submission.csv` |

**How to use:** Settings -> Internet **off**, Accelerator GPU T4. Edit the SETTINGS block in section 1, then *Save Version -> Save & Run All (Commit)*. Read the table printed in section 7 first: it tells you which detector mode and which `K_MAD` / `SWIN_THRESHOLD` work best on movies the model never saw.

## 1. Settings, paths, offline-safe imports

In [1]:
# !pip install --no-index --find-links=/kaggle/input/notebooks/parth2157/zarr-wheels zarr
!pip install --no-index --find-links=/kaggle/input/notebooks/parth2157/zarr-wheels/zarr_wheels zarr


Looking in links: /kaggle/input/notebooks/parth2157/zarr-wheels/zarr_wheels
Processing /kaggle/input/notebooks/parth2157/zarr-wheels/zarr_wheels/zarr-3.4.2-py3-none-any.whl
Processing /kaggle/input/notebooks/parth2157/zarr-wheels/zarr_wheels/donfig-0.8.1.post1-py3-none-any.whl (from zarr)
Processing /kaggle/input/notebooks/parth2157/zarr-wheels/zarr_wheels/msgspec-0.22.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (from zarr)
Processing /kaggle/input/notebooks/parth2157/zarr-wheels/zarr_wheels/numcodecs-0.17.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (from zarr)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [zarr]


In [2]:
import os, sys, time, math, random, subprocess, traceback, fnmatch, importlib
from pathlib import Path
from collections import Counter, defaultdict
from importlib.metadata import version as _pkg_version, PackageNotFoundError
import numpy as np
import pandas as pd
import torch

# ============================ SETTINGS ============================
DETECTOR_MODE = "hybrid"   # "classical" = blob detector only (no training) | "swin" = Swin net only | "hybrid" = both merged
TRAIN_MINUTES = 0          # Swin training time inside this notebook. 0 = skip (uses attached weights if you added any)
USE_PSEUDO_LABELS = True   # also train on blob-detector peaks, so unannotated nuclei are not taught as "background"
SWIN_THRESHOLD = 0.3       # lower = more Swin detections
K_MAD = 4.0                # blob detector sensitivity: lower = more detections (AUTO_TUNE below may change it)
MAX_LINK_UM = 10.0         # how far (micrometres) a cell may move between two frames (AUTO_TUNE may change it)
AUTO_TUNE = True           # section 7b: pick K_MAD and MAX_LINK_UM on the held-out movies
N_HOLDOUT = 2              # training movies kept out of training and used only to measure the pipeline
CHECK_FRAMES = 8           # frames of each held-out movie used for that measurement
# ==================================================================

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [Path(f"/kaggle/input/competitions/{COMPETITION}"), Path(f"/kaggle/input/{COMPETITION}")]
COMP_DIR = next((p for p in COMP_DIR_CANDIDATES if p.exists()), COMP_DIR_CANDIDATES[0])
TRAIN_DIR, TEST_DIR = COMP_DIR / "train", COMP_DIR / "test"
SAMPLE_SUBMISSION = COMP_DIR / "sample_submission.csv"
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
WEIGHTS_PATH = WORKING_DIR / "swin_detector.pt"
print("COMP_DIR:", COMP_DIR, "| exists:", COMP_DIR.exists(), "| train:", TRAIN_DIR.exists(), "| test:", TEST_DIR.exists())


def find_files(root, patterns, max_depth=6, skip_names=()):
    """Walk `root` (never into movie folders) and return files matching any of `patterns`."""
    out, root = [], str(root)
    if not os.path.isdir(root):
        return out
    base = root.rstrip(os.sep).count(os.sep)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.count(os.sep) - base
        dirnames[:] = [d for d in dirnames if d not in skip_names and not d.endswith((".zarr", ".geff")) and depth < max_depth]
        out += [os.path.join(dirpath, f) for f in filenames if any(fnmatch.fnmatch(f, p) for p in patterns)]
    return out


def ensure_zarr():
    """Competition notebooks run with Internet OFF. Need zarr >= 3: use it if installed, else look for attached wheels."""
    def ok():
        try:
            return int(_pkg_version("zarr").split(".")[0]) >= 3
        except PackageNotFoundError:
            return False
    try:
        print("installed zarr:", _pkg_version("zarr"), "(need >= 3)")
    except PackageNotFoundError:
        print("installed zarr: none (need >= 3)")
    if ok():
        return
    wheel_dirs = sorted({os.path.dirname(w) for w in find_files("/kaggle/input", ["zarr-*.whl"], skip_names=(COMPETITION,))})
    print("folders with zarr wheels under /kaggle/input:", wheel_dirs or "none -> attach the zarr-wheels dataset (Add Input)")
    for d in wheel_dirs:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", d, "zarr>=3.0.10,<4"],
                           capture_output=True, text=True)
        importlib.invalidate_caches()
        print("offline zarr install from", d, "->", "ok" if r.returncode == 0 else r.stderr[-200:])
        if ok():
            return
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--retries", "1", "--timeout", "10", "zarr>=3.0.10,<4"],
                   capture_output=True, text=True)          # only works if Internet is ON (interactive runs)
    importlib.invalidate_caches()
    if not ok():
        raise RuntimeError("zarr >= 3 is not available and no usable zarr wheels were found. Create a dataset named zarr-wheels "
                           "(pip download \"zarr>=3.0.10,<4\" \"numcodecs<0.16\" --only-binary=:all: -d zarr_wheels, run with Internet ON), "
                           "attach it with Add Input, and run again with Internet OFF.")

ensure_zarr()
import zarr
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| zarr", zarr.__version__, "| device:", device)


COMP_DIR: /kaggle/input/competitions/biohub-cell-tracking-during-development | exists: True | train: True | test: True
installed zarr: 3.4.2 (need >= 3)
torch 2.11.0+cu130 | zarr 3.4.2 | device: cuda


## 2. Swin detector model and tiled inference

Pure PyTorch (no MONAI), unchanged from before except that inference now covers the **last frame** of every movie (the old loop skipped it).

In [3]:
from __future__ import annotations

from typing import Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock3D(nn.Module):
    """2 x (Conv 3x3x3, padding 1 -> BatchNorm -> ReLU). Same block as the CNN U-Net."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class DropPath(nn.Module):
    """Stochastic depth (per sample)."""

    def __init__(self, p: float = 0.0):
        super().__init__()
        self.p = p

    def forward(self, x):
        if self.p == 0.0 or not self.training:
            return x
        keep = 1.0 - self.p
        mask = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(keep)
        return x * mask / keep


def window_partition(x: torch.Tensor, ws: Tuple[int, int, int]) -> torch.Tensor:
    """(B, Z, Y, X, C) -> (B * nWindows, wz * wy * wx, C). Z, Y, X must be divisible by ws."""
    B, Z, Y, X, C = x.shape
    wz, wy, wx = ws
    x = x.view(B, Z // wz, wz, Y // wy, wy, X // wx, wx, C)
    x = x.permute(0, 1, 3, 5, 2, 4, 6, 7).contiguous()
    return x.view(-1, wz * wy * wx, C)


def window_reverse(w: torch.Tensor, ws: Tuple[int, int, int], dims: Tuple[int, int, int, int]) -> torch.Tensor:
    """Inverse of window_partition. dims = (B, Z, Y, X)."""
    B, Z, Y, X = dims
    wz, wy, wx = ws
    C = w.shape[-1]
    x = w.view(B, Z // wz, Y // wy, X // wx, wz, wy, wx, C)
    x = x.permute(0, 1, 4, 2, 5, 3, 6, 7).contiguous()
    return x.view(B, Z, Y, X, C)


def build_shift_mask(grid: Tuple[int, int, int], ws: Tuple[int, int, int],
                     shift: Tuple[int, int, int]) -> torch.Tensor:
    """Attention mask (nWindows, N, N) that stops tokens attending across the cyclic-shift seam."""
    def slices(length, w, s):
        if s == 0:
            return (slice(None),)
        return (slice(0, -w), slice(-w, -s), slice(-s, None))

    img = torch.zeros((1,) + tuple(grid) + (1,))
    cnt = 0
    for zs in slices(grid[0], ws[0], shift[0]):
        for ys in slices(grid[1], ws[1], shift[1]):
            for xs in slices(grid[2], ws[2], shift[2]):
                img[:, zs, ys, xs, :] = cnt
                cnt += 1
    win = window_partition(img, ws).squeeze(-1)        # (nW, N)
    diff = win.unsqueeze(1) - win.unsqueeze(2)         # (nW, N, N)
    return diff.masked_fill(diff != 0, -100.0).masked_fill(diff == 0, 0.0)


class WindowAttention3D(nn.Module):
    """Multi-head self-attention inside 3D windows, with a learned relative position bias."""

    def __init__(self, dim: int, window: Tuple[int, int, int], num_heads: int):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        self.heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        wz, wy, wx = window
        self.rel_bias_table = nn.Parameter(torch.zeros((2 * wz - 1) * (2 * wy - 1) * (2 * wx - 1), num_heads))
        nn.init.trunc_normal_(self.rel_bias_table, std=0.02)

        coords = torch.stack(torch.meshgrid(torch.arange(wz), torch.arange(wy), torch.arange(wx), indexing="ij"))
        flat = coords.flatten(1)                                   # (3, N)
        rel = (flat[:, :, None] - flat[:, None, :]).permute(1, 2, 0).contiguous()   # (N, N, 3)
        rel[..., 0] += wz - 1
        rel[..., 1] += wy - 1
        rel[..., 2] += wx - 1
        rel[..., 0] *= (2 * wy - 1) * (2 * wx - 1)
        rel[..., 1] *= (2 * wx - 1)
        self.register_buffer("rel_index", rel.sum(-1), persistent=False)   # (N, N)

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        nB, N, C = x.shape
        qkv = self.qkv(x).reshape(nB, N, 3, self.heads, C // self.heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q * self.scale) @ k.transpose(-2, -1)              # (nB, heads, N, N)
        bias = self.rel_bias_table[self.rel_index.view(-1)].view(N, N, -1).permute(2, 0, 1)
        attn = attn + bias.unsqueeze(0)
        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(nB // nW, nW, self.heads, N, N) + mask.unsqueeze(1).unsqueeze(0).to(attn.dtype)
            attn = attn.view(nB, self.heads, N, N)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(nB, N, C)
        return self.proj(out)


class SwinBlock3D(nn.Module):
    """LayerNorm -> (shifted) window attention -> residual; LayerNorm -> MLP -> residual."""

    def __init__(self, dim, grid, window, shift, heads, mlp_ratio=4.0, drop_path=0.0):
        super().__init__()
        self.window, self.shift = tuple(window), tuple(shift)
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention3D(dim, self.window, heads)
        self.drop_path = DropPath(drop_path)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))
        if any(self.shift):
            self.register_buffer("attn_mask", build_shift_mask(grid, self.window, self.shift), persistent=False)
        else:
            self.attn_mask = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:           # x: (B, Z, Y, X, C)
        B, Z, Y, X, _ = x.shape
        shortcut = x
        x = self.norm1(x)
        if any(self.shift):
            x = torch.roll(x, shifts=tuple(-s for s in self.shift), dims=(1, 2, 3))
        w = self.attn(window_partition(x, self.window), self.attn_mask)
        x = window_reverse(w, self.window, (B, Z, Y, X))
        if any(self.shift):
            x = torch.roll(x, shifts=self.shift, dims=(1, 2, 3))
        x = shortcut + self.drop_path(x)
        return x + self.drop_path(self.mlp(self.norm2(x)))


class SwinStage3D(nn.Module):
    """`depth` Swin blocks; odd blocks use shifted windows (on axes where the window is smaller than the grid)."""

    def __init__(self, dim, grid, window, depth, heads, mlp_ratio, drop_paths):
        super().__init__()
        win = tuple(min(w, g) for w, g in zip(window, grid))
        for g, w in zip(grid, win):
            if g % w != 0:
                raise ValueError(f"grid {grid} is not divisible by window {win}; change img_size or window_size")
        shift = tuple((w // 2) if w < g else 0 for w, g in zip(win, grid))
        self.blocks = nn.ModuleList([
            SwinBlock3D(dim, grid, win, (0, 0, 0) if i % 2 == 0 else shift, heads, mlp_ratio, drop_paths[i])
            for i in range(depth)
        ])

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        return x


class PatchEmbed3D(nn.Module):
    def __init__(self, in_ch, dim, patch):
        super().__init__()
        self.proj = nn.Conv3d(in_ch, dim, kernel_size=patch, stride=patch)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):                                          # (B, C, Z, Y, X) -> (B, z, y, x, dim)
        return self.norm(self.proj(x).permute(0, 2, 3, 4, 1))


class PatchMerging3D(nn.Module):
    """Concatenate each 2x2x2 neighbourhood (8C) and project to 2C: halves Z, Y, X."""

    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(8 * dim)
        self.reduction = nn.Linear(8 * dim, 2 * dim, bias=False)

    def forward(self, x):                                          # (B, Z, Y, X, C)
        parts = [x[:, i::2, j::2, k::2] for i in (0, 1) for j in (0, 1) for k in (0, 1)]
        return self.reduction(self.norm(torch.cat(parts, dim=-1)))


class SwinDetector3D(nn.Module):
    def __init__(
        self,
        in_channels: int = 1,
        img_size: Sequence[int] = (16, 96, 96),
        patch_size: Sequence[int] = (2, 2, 2),
        embed_dim: int = 24,
        depths: Sequence[int] = (2, 2, 2),
        num_heads: Sequence[int] = (3, 6, 12),
        window_size: Sequence[int] = (2, 6, 6),
        mlp_ratio: float = 4.0,
        drop_path: float = 0.1,
        stem_channels: int = 16,
        n_temporal_heads: int = 4,
    ):
        super().__init__()
        assert len(depths) == len(num_heads)
        self.img_size = tuple(img_size)
        self.feature_dim = stem_channels
        n = len(depths)
        dims = [embed_dim * 2 ** i for i in range(n)]               # 24, 48, 96
        grid = tuple(s // p for s, p in zip(self.img_size, patch_size))
        for s, p in zip(self.img_size, patch_size):
            if s % (p * 2 ** (n - 1)) != 0:
                raise ValueError(f"img_size {self.img_size} must be divisible by patch_size * 2**{n - 1}")

        # encoder
        self.stem = ConvBlock3D(in_channels, stem_channels)
        self.patch_embed = PatchEmbed3D(in_channels, dims[0], tuple(patch_size))
        dp = torch.linspace(0, drop_path, sum(depths)).tolist()
        self.stages, self.merges = nn.ModuleList(), nn.ModuleList()
        k = 0
        for i in range(n):
            self.stages.append(SwinStage3D(dims[i], grid, tuple(window_size), depths[i], num_heads[i],
                                           mlp_ratio, dp[k:k + depths[i]]))
            k += depths[i]
            if i < n - 1:
                self.merges.append(PatchMerging3D(dims[i]))
                grid = tuple(g // 2 for g in grid)

        # temporal attention gate on the bottleneck (same idea as the CNN U-Net)
        self.temporal_attn = nn.MultiheadAttention(embed_dim=dims[-1], num_heads=n_temporal_heads, batch_first=True)

        # conv decoder with skip connections
        self.ups, self.decs = nn.ModuleList(), nn.ModuleList()
        for i in range(n - 1, 0, -1):
            self.ups.append(nn.ConvTranspose3d(dims[i], dims[i - 1], kernel_size=2, stride=2))
            self.decs.append(ConvBlock3D(2 * dims[i - 1], dims[i - 1]))
        self.up_final = nn.ConvTranspose3d(dims[0], stem_channels, kernel_size=tuple(patch_size), stride=tuple(patch_size))
        self.dec_final = ConvBlock3D(2 * stem_channels, stem_channels)
        self.head = nn.Conv3d(stem_channels, 1, kernel_size=1)

    def forward(self, x: torch.Tensor):
        B, T, C, Z, Y, X = x.shape
        if (Z, Y, X) != self.img_size:
            raise ValueError(f"input spatial size {(Z, Y, X)} != img_size {self.img_size} this model was built for")
        xf = x.reshape(B * T, C, Z, Y, X)

        stem = self.stem(xf)
        t = self.patch_embed(xf)
        skips = []
        for i, stage in enumerate(self.stages):
            t = stage(t)
            if i < len(self.stages) - 1:
                skips.append(t.permute(0, 4, 1, 2, 3).contiguous())
                t = self.merges[i](t)
        b = t.permute(0, 4, 1, 2, 3).contiguous()                  # (B*T, 96, 2, 12, 12)

        pooled = b.mean(dim=(2, 3, 4)).reshape(B, T, -1)           # (B, T, 96)
        gate, _ = self.temporal_attn(pooled, pooled, pooled)
        d = b * torch.sigmoid(gate).reshape(B * T, -1, 1, 1, 1)

        for up, dec, skip in zip(self.ups, self.decs, reversed(skips)):
            d = dec(torch.cat([up(d), skip], dim=1))
        d = self.dec_final(torch.cat([self.up_final(d), stem], dim=1))   # (B*T, 16, Z, Y, X)

        heatmap = self.head(d).reshape(B, T, 1, Z, Y, X)
        features = d.reshape(B, T, self.feature_dim, Z, Y, X)
        return heatmap, features


def build_detector(config: dict) -> nn.Module:
    """Pick the detector from CONFIG. Set CONFIG['detector'] = 'swin' or 'cnn'."""
    if config.get("detector", "cnn") == "swin":
        return SwinDetector3D(img_size=tuple(config["patch_size"]),
                              embed_dim=config.get("swin_embed_dim", 24),
                              n_temporal_heads=config.get("n_heads", 4))
    # 'cnn' needs Detector3DUNetTemporal from your training notebook to be defined
    return Detector3DUNetTemporal(base_channels=config["base_channels"], n_heads=config["n_heads"])  # noqa: F821

In [4]:
def tile_starts(length: int, size: int):
    if length <= size:
        return [0]
    starts = list(range(0, length - size + 1, size))
    if starts[-1] + size < length:
        starts.append(length - size)
    return starts


def owned_range(starts, i: int, size: int):
    lo, hi = starts[i], starts[i] + size
    if i > 0:
        lo = max(lo, starts[i - 1] + size)
    return lo, hi


def detect_and_pool_streaming(image, detector, device, patch_size, t_window, voxel_scale,
                              heatmap_threshold=0.5, min_distance_um=3.0, return_features=False):
    """Same signature as before; works with the CNN or the Swin detector and covers every voxel."""
    import numpy as np
    import pandas as pd
    from skimage.feature import peak_local_max

    T, Z, Y, X = image.shape
    pz, py, px = patch_size
    zs, ys, xs = tile_starts(Z, pz), tile_starts(Y, py), tile_starts(X, px)
    min_dist = max(1, int(round(min_distance_um / float(np.min(voxel_scale)))))

    records, feats = [], {t: [] for t in range(T)}
    next_id = 1
    detector.eval()
    with torch.no_grad():
        for t0 in range(0, T, t_window):
            actual_t = min(t0 + t_window, T) - t0
            for iz, z0 in enumerate(zs):
                zlo, zhi = owned_range(zs, iz, pz)
                for iy, y0 in enumerate(ys):
                    ylo, yhi = owned_range(ys, iy, py)
                    for ix, x0 in enumerate(xs):
                        xlo, xhi = owned_range(xs, ix, px)
                        crop = np.asarray(image[t0:t0 + t_window, z0:z0 + pz, y0:y0 + py, x0:x0 + px]).astype(np.float32)
                        if crop.shape[0] < t_window:
                            pad = np.zeros((t_window - crop.shape[0],) + crop.shape[1:], dtype=np.float32)
                            crop = np.concatenate([crop, pad], axis=0)
                        crop = (crop - crop.mean()) / (crop.std() + 1e-6)
                        inp = torch.from_numpy(crop).unsqueeze(0).unsqueeze(2).to(device)
                        heat, feat = detector(inp)
                        heat = torch.sigmoid(heat).cpu().numpy()[0, :, 0]
                        feat = feat.cpu().numpy()[0] if return_features else None
                        for lt in range(actual_t):
                            t = t0 + lt
                            for z, y, x in peak_local_max(heat[lt], threshold_abs=heatmap_threshold,
                                                          min_distance=min_dist, exclude_border=False):
                                gz, gy, gx = z0 + int(z), y0 + int(y), x0 + int(x)
                                if not (zlo <= gz < zhi and ylo <= gy < yhi and xlo <= gx < xhi):
                                    continue
                                records.append({"node_id": next_id, "t": t, "z": gz, "y": gy, "x": gx})
                                if return_features:
                                    feats[t].append(feat[lt, :, z, y, x].copy())
                                next_id += 1
    features_by_t = ({t: (np.stack(f) if f else np.zeros((0, detector.feature_dim), dtype=np.float32))
                      for t, f in feats.items()} if return_features else {})
    return pd.DataFrame(records), features_by_t

In [5]:
CONFIG = {"detector": "swin", "patch_size": (16, 96, 96), "swin_embed_dim": 24, "n_heads": 4}


def list_stems(d):
    return sorted(p.name[:-5] for p in Path(d).glob("*.zarr")) if Path(d).exists() else []


def open_movie(split_dir, stem):
    """Returns (zarr array (T,Z,Y,X), voxel_scale (z,y,x) in micrometres)."""
    store = zarr.open(str(Path(split_dir) / f"{stem}.zarr"), mode="r")
    scale = store.attrs["multiscales"][0]["datasets"][0]["coordinateTransformations"][0]["scale"]
    return store["0"], tuple(scale[1:])


def load_geff_graph(geff_path):
    """Load a .geff annotation graph into (nodes_df, edges_df)."""
    store = zarr.open(str(geff_path), mode="r")
    nodes_df = pd.DataFrame({"node_id": store["nodes/ids"][:], "t": store["nodes/props/t/values"][:],
                             "z": store["nodes/props/z/values"][:], "y": store["nodes/props/y/values"][:],
                             "x": store["nodes/props/x/values"][:]})
    ids = store["edges/ids"][:]
    return nodes_df, pd.DataFrame({"source_id": ids[:, 0], "target_id": ids[:, 1]})


class FirstFrames:
    """View of the first n frames of a (T,Z,Y,X) array, so quick checks do not process the whole movie."""
    def __init__(self, arr, n):
        self.arr, self.shape = arr, (min(n, arr.shape[0]),) + tuple(arr.shape[1:])
    def __getitem__(self, idx):
        return self.arr[idx]


## 3. Blob detector (needs no training)

Difference of Gaussians in micrometre units (z is coarser than y/x), then local maxima above a noise-based threshold. The metric does not punish extra nodes (unmatched nodes are not false positives), so catching nearly every nucleus is what matters. Tune it with `K_MAD` in section 1.

In [6]:
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components


def classical_peaks(vol, scale, sigma_small_um=2.0, sigma_large_um=4.5, min_dist_um=3.0, k_mad=4.0, max_peaks=6000):
    """Blob detector for one 3D frame: difference of Gaussians, then local maxima above a noise-based threshold.
    Returns an (N,3) integer array of (z,y,x) voxel positions. No training needed."""
    v = np.asarray(vol, dtype=np.float32)
    lo, hi = np.percentile(v, [1.0, 99.8])
    v = (v - lo) / max(float(hi - lo), 1e-6)
    s_small = [sigma_small_um / s for s in scale]
    s_large = [sigma_large_um / s for s in scale]
    dog = gaussian_filter(v, s_small, truncate=3.0) - gaussian_filter(v, s_large, truncate=3.0)
    med = float(np.median(dog))
    mad = float(np.median(np.abs(dog - med))) * 1.4826 + 1e-6
    size = tuple(2 * max(int(round(min_dist_um / s)), 1) + 1 for s in scale)
    is_peak = (dog == maximum_filter(dog, size=size)) & (dog > med + k_mad * mad)
    pk = np.argwhere(is_peak)
    if len(pk) > max_peaks:
        pk = pk[np.argsort(-dog[tuple(pk.T)])[:max_peaks]]
    return pk


def points_to_nodes(points_by_t):
    """{t: (N,3) voxel array} -> nodes_df with unique node_id (starting at 1)."""
    rows, nid = [], 1
    for t in sorted(points_by_t):
        for z, y, x in points_by_t[t]:
            rows.append((nid, int(t), int(z), int(y), int(x)))
            nid += 1
    return pd.DataFrame(rows, columns=["node_id", "t", "z", "y", "x"], dtype="int64")


def classical_detect_movie(movie, scale, k_mad=None, verbose=True):
    k_mad = K_MAD if k_mad is None else k_mad
    pts = {}
    for t in range(movie.shape[0]):
        pts[t] = classical_peaks(movie[t], scale, k_mad=k_mad)
        if verbose and (t + 1) % 25 == 0:
            print(f"   blob detector: frame {t + 1}/{movie.shape[0]}")
    return points_to_nodes(pts)


def merge_detections(primary, secondary, scale, min_dist_um=3.0):
    """Keep every `primary` node; add `secondary` nodes that are farther than min_dist_um from all primary nodes of that frame."""
    vs = np.asarray(scale, dtype=float)
    parts = [primary[["t", "z", "y", "x"]]]
    sec_by_t = {int(t): g for t, g in secondary.groupby("t")}
    pri_by_t = {int(t): g for t, g in primary.groupby("t")}
    for t, g in sec_by_t.items():
        if t not in pri_by_t:
            parts.append(g[["t", "z", "y", "x"]])
            continue
        d, _ = cKDTree(pri_by_t[t][["z", "y", "x"]].to_numpy() * vs).query(g[["z", "y", "x"]].to_numpy() * vs)
        parts.append(g.loc[d > min_dist_um, ["t", "z", "y", "x"]])
    out = pd.concat(parts, ignore_index=True).sort_values("t", kind="stable").reset_index(drop=True)
    out.insert(0, "node_id", np.arange(1, len(out) + 1))
    return out.astype("int64")


## 4. Train the Swin detector (optional, falls back safely)

Targets = annotated nuclei **plus blob-detector peaks** (`USE_PSEUDO_LABELS`), so real but unannotated nuclei are not taught as background. Annotated nuclei get the highest loss weight. If you attach a dataset that contains `swin_detector*.pt`, it is loaded and training is skipped. Set `TRAIN_MINUTES = 0` to skip training.

In [7]:
def _blob(delta, sigma):
    """Turn a 0/1 point volume (T,Z,Y,X) into Gaussian blobs with peak 1 (sigma in voxels)."""
    peak = gaussian_filter(np.pad(np.ones((1, 1, 1), np.float32), 8), sigma, mode="constant").max()
    out = np.empty_like(delta)
    for i in range(delta.shape[0]):
        out[i] = np.minimum(gaussian_filter(delta[i], sigma, mode="constant") / peak, 1.0)
    return out


class CropSampler:
    """Random training crops near an annotated nucleus.
    gt     = Gaussian blobs at the annotated nuclei (reliable, but only ~3% of cells)
    pseudo = Gaussian blobs at blob-detector peaks (covers the cells nobody annotated)"""

    def __init__(self, movies, patch=(16, 96, 96), n_frames=4, sigma_um=2.0, pseudo=True):
        self.movies, self.patch, self.T, self.sigma_um, self.pseudo = movies, tuple(patch), n_frames, sigma_um, pseudo
        self.keys = list(movies)

    def sample(self):
        pz, py, px = self.patch
        m = self.movies[random.choice(self.keys)]
        nodes = m["nodes"]
        r = nodes.iloc[random.randrange(len(nodes))]
        Tn, Zn, Yn, Xn = m["img"].shape
        T = min(self.T, Tn)
        t0 = int(np.clip(int(r.t) - random.randint(0, T - 1), 0, Tn - T))

        def start(c, size, n):
            return int(np.clip(int(c) - random.randint(size // 4, 3 * size // 4), 0, max(n - size, 0)))

        z0, y0, x0 = start(r.z, pz, Zn), start(r.y, py, Yn), start(r.x, px, Xn)
        crop = np.asarray(m["img"][t0:t0 + T, z0:z0 + pz, y0:y0 + py, x0:x0 + px]).astype(np.float32)
        sel = nodes[(nodes.t >= t0) & (nodes.t < t0 + T) & (nodes.z >= z0) & (nodes.z < z0 + pz)
                    & (nodes.y >= y0) & (nodes.y < y0 + py) & (nodes.x >= x0) & (nodes.x < x0 + px)]
        gt = np.zeros(crop.shape, np.float32)
        for row in sel.itertuples():
            gt[int(row.t) - t0, int(row.z) - z0, int(row.y) - y0, int(row.x) - x0] = 1.0
        sigma = [self.sigma_um / s for s in m["scale"]]
        gt = _blob(gt, sigma)
        ps = np.zeros_like(gt)
        if self.pseudo:
            for i in range(T):
                pk = classical_peaks(crop[i], m["scale"], k_mad=K_MAD, max_peaks=2000)
                if len(pk):
                    ps[i][pk[:, 0], pk[:, 1], pk[:, 2]] = 1.0
            ps = _blob(ps, sigma)
        crop = (crop - crop.mean()) / (crop.std() + 1e-6)          # same normalisation as inference
        if random.random() < 0.5:
            crop, gt, ps = crop[:, :, ::-1], gt[:, :, ::-1], ps[:, :, ::-1]
        if random.random() < 0.5:
            crop, gt, ps = crop[..., ::-1], gt[..., ::-1], ps[..., ::-1]
        if T < self.T:
            pad = ((0, self.T - T),) + ((0, 0),) * 3
            crop, gt, ps = np.pad(crop, pad), np.pad(gt, pad), np.pad(ps, pad)
        return [np.ascontiguousarray(a) for a in (crop, gt, ps)]

    def batch(self, size):
        xs, gs, ps = zip(*(self.sample() for _ in range(size)))
        f = lambda a: torch.from_numpy(np.stack(a)).unsqueeze(2)    # (B,T,1,Z,Y,X)
        return f(xs), f(gs), f(ps)


def train_detector(model, sampler, minutes=15, batch_size=2, lr=3e-4, log_every=50, save_every=500):
    """Time-budgeted training with cosine learning-rate decay."""
    import torch.nn.functional as F
    use_amp = device.type == "cuda"
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    budget, t_start, step, running = minutes * 60.0, time.time(), 0, []
    model.train()
    while time.time() - t_start < budget:
        frac = (time.time() - t_start) / budget
        for g in opt.param_groups:
            g["lr"] = lr * 0.5 * (1 + math.cos(math.pi * frac))
        x, gt, ps = (a.to(device) for a in sampler.batch(batch_size))
        target = torch.maximum(gt, 0.8 * ps)
        weight = 1.0 + 20.0 * gt + 4.0 * ps                       # annotated nuclei count most
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits, _ = model(x)
        loss = (F.binary_cross_entropy_with_logits(logits.float(), target, reduction="none") * weight).mean()
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        running.append(loss.item())
        step += 1
        if step % log_every == 0:
            print(f"step {step:5d} | loss {np.mean(running[-log_every:]):.4f} | {(time.time() - t_start) / 60:.1f} min")
        if step % save_every == 0:
            torch.save({"config": CONFIG, "state_dict": model.state_dict()}, WEIGHTS_PATH)
    torch.save({"config": CONFIG, "state_dict": model.state_dict()}, WEIGHTS_PATH)
    print(f"training done: {step} steps -> {WEIGHTS_PATH}")
    model.eval()
    return model


In [8]:
train_stems = list_stems(TRAIN_DIR)
fit_stems = train_stems[:-N_HOLDOUT] if len(train_stems) > N_HOLDOUT else train_stems
holdout_stems = train_stems[-N_HOLDOUT:] if len(train_stems) > N_HOLDOUT else []
print("train movies:", len(train_stems), "| training on", len(fit_stems), "| held out:", holdout_stems)

swin_model, SWIN_READY = None, False
if DETECTOR_MODE in ("swin", "hybrid"):
    try:
        # 1) weights from an attached dataset win (train once, reuse in every later run)
        attached = find_files("/kaggle/input", ["swin_detector*.pt"], skip_names=(COMPETITION,))
        if attached:
            ckpt = torch.load(attached[0], map_location=device)
            swin_model = build_detector(ckpt["config"]).to(device)
            swin_model.load_state_dict(ckpt["state_dict"])
            swin_model.eval()
            SWIN_READY = True
            print("loaded attached weights:", attached[0])
        # 2) otherwise train here
        elif TRAIN_MINUTES > 0 and fit_stems and (TRAIN_DIR / f"{fit_stems[0]}.geff").exists():
            movies = {}
            for s in fit_stems:
                img, scale = open_movie(TRAIN_DIR, s)
                nodes, _ = load_geff_graph(TRAIN_DIR / f"{s}.geff")
                movies[s] = {"img": img, "nodes": nodes, "scale": scale}
                print(f"  {s}: shape {img.shape}, {len(nodes):,} annotated nodes")
            sampler = CropSampler(movies, patch=CONFIG["patch_size"], n_frames=4, pseudo=USE_PSEUDO_LABELS)
            x0, g0, p0 = sampler.batch(2)
            print("batch:", tuple(x0.shape), "| annotated-blob voxels:", float((g0 > 0.5).float().mean()),
                  "| pseudo-blob voxels:", float((p0 > 0.5).float().mean()))
            swin_model = build_detector(CONFIG).to(device)
            swin_model = train_detector(swin_model, sampler, minutes=TRAIN_MINUTES, batch_size=2 if device.type == "cuda" else 1)
            SWIN_READY = True
        else:
            print("No weights attached and training skipped -> the Swin net is not used.")
    except Exception:
        traceback.print_exc()
        print("Swin part failed -> continuing with the blob detector only.")
        swin_model, SWIN_READY = None, False
print("SWIN_READY:", SWIN_READY)


train movies: 199 | training on 197 | held out: ['6bba_fc83837d', '6bba_fe670320']
No weights attached and training skipped -> the Swin net is not used.
SWIN_READY: False


## 5. Detection wrapper

`classical`, `swin` or `hybrid` (blob detections + extra Swin detections). Anything that fails falls back to the blob detector.

In [9]:
def swin_detect_movie(movie, scale):
    nodes, _ = detect_and_pool_streaming(movie, swin_model, device, CONFIG["patch_size"], t_window=4, voxel_scale=scale,
                                         heatmap_threshold=SWIN_THRESHOLD, min_distance_um=3.0)
    return nodes[["node_id", "t", "z", "y", "x"]].astype("int64")


def detect_movie(movie, scale, mode=None):
    """mode: "classical" | "swin" | "hybrid". Falls back to the blob detector when the Swin net is not ready or fails."""
    mode = mode or DETECTOR_MODE
    want_swin = mode in ("swin", "hybrid") and SWIN_READY
    if mode in ("swin", "hybrid") and not SWIN_READY:
        print("   (Swin not ready -> blob detector only)")
    if mode == "swin" and want_swin:
        try:
            return swin_detect_movie(movie, scale)
        except Exception:
            traceback.print_exc()
    nodes = classical_detect_movie(movie, scale)
    if mode == "hybrid" and want_swin:
        try:
            nodes = merge_detections(nodes, swin_detect_movie(movie, scale), scale)
        except Exception:
            traceback.print_exc()
    return nodes


## 6. Linking: Hungarian matching + velocity, safe divisions, gap closing, and a local metric

* Frame t to t+1 matching minimises total distance (one-to-one), after predicting each cell's position from its last move.
* A second daughter is added only when it is very close to the parent *and* to its sister, with small caps (a wrong division costs more than a missed one).
* A cell that vanishes for exactly one frame is bridged with a synthetic node.
* `local_edge_metrics` re-implements the edge rules of the official `metrics.md`.

In [10]:
def match_frames(pa, pb, max_um, dense_limit=4_000_000):
    """One-to-one matching of points pa (Na,3) to pb (Nb,3) in micrometres, total distance minimal, gate max_um.
    Splits the problem into independent clusters first, so it stays fast with thousands of cells per frame."""
    Na, Nb = len(pa), len(pb)
    if Na == 0 or Nb == 0:
        return []
    coo = cKDTree(pb).sparse_distance_matrix(cKDTree(pa), max_um, output_type="coo_matrix")
    if coo.nnz == 0:
        return []
    rb, ca, d = coo.row.astype(np.int64), coo.col.astype(np.int64), coo.data + 1e-9
    graph = csr_matrix((np.ones(len(d)), (ca, Na + rb)), shape=(Na + Nb, Na + Nb))
    _, lab = connected_components(graph, directed=False)
    comp = lab[ca]
    order = np.argsort(comp, kind="stable")
    cuts = np.flatnonzero(np.diff(comp[order])) + 1
    pairs = []
    for idx in np.split(order, cuts):
        a_u, a_inv = np.unique(ca[idx], return_inverse=True)
        b_u, b_inv = np.unique(rb[idx], return_inverse=True)
        if len(a_u) * len(b_u) <= dense_limit:
            big = 1e9
            cost = np.full((len(a_u), len(b_u)), big)
            cost[a_inv, b_inv] = d[idx]
            r, c = linear_sum_assignment(cost)
            pairs += [(int(a_u[i]), int(b_u[j])) for i, j in zip(r, c) if cost[i, j] < big]
        else:                                                        # huge cluster: greedy
            used_a, used_b = set(), set()
            for k in idx[np.argsort(d[idx])]:
                if ca[k] not in used_a and rb[k] not in used_b:
                    used_a.add(ca[k]); used_b.add(rb[k]); pairs.append((int(ca[k]), int(rb[k])))
    return pairs


def link_tracks(nodes_df, scale, max_link_um=None, vel_weight=0.5,
                divisions=True, div_parent_um=4.6, div_sister_um=6.9, div_frame_frac=0.008, div_global_frac=0.0036,
                gap_close=True, gap_um=5.9, gap_max_frac=0.045):
    """Frame-to-frame linking with constant-velocity prediction, then safe divisions, then 1-frame gap closing.
    Returns (nodes_df incl. synthetic gap nodes, edges_df). Every edge goes from frame t to t+1."""
    max_link_um = MAX_LINK_UM if max_link_um is None else max_link_um
    vs = np.asarray(scale, dtype=float)
    nodes = nodes_df[["node_id", "t", "z", "y", "x"]].astype("int64").sort_values(["t", "node_id"]).reset_index(drop=True)
    N = len(nodes)
    by_t = {}
    for t, g in nodes.groupby("t"):
        by_t[int(t)] = (g["node_id"].to_numpy(), g[["z", "y", "x"]].to_numpy() * vs)
    pos_of = {int(i): p for ids, ps in by_t.values() for i, p in zip(ids, ps)}
    parent, edges, div_cands = {}, [], []

    for t in sorted(by_t):
        if t + 1 not in by_t:
            continue
        ida, pa = by_t[t]
        idb, pb = by_t[t + 1]
        pred = pa.copy()
        for i, nid in enumerate(ida):
            p = parent.get(int(nid))
            if p is not None:
                pred[i] = pa[i] + vel_weight * (pa[i] - pos_of[p])
        pairs = match_frames(pred, pb, max_link_um)
        child_of = {}
        for i, j in pairs:
            edges.append((int(ida[i]), int(idb[j])))
            parent[int(idb[j])] = int(ida[i])
            child_of[i] = j
        if divisions and child_of:                                   # collect division candidates for this frame
            matched_b = set(child_of.values())
            free_b = [j for j in range(len(idb)) if j not in matched_b]
            if free_b:
                a_idx = np.array(sorted(child_of))
                d, k = cKDTree(pa[a_idx]).query(pb[free_b], distance_upper_bound=div_parent_um)
                best = {}
                for q, (dd, kk) in enumerate(zip(d, k)):
                    if not np.isfinite(dd):
                        continue
                    i, j = int(a_idx[kk]), free_b[q]
                    if np.linalg.norm(pb[j] - pb[child_of[i]]) <= div_sister_um and (i not in best or dd < best[i][0]):
                        best[i] = (dd, j)                           # one extra daughter per parent
                frame_cap = max(1, int(div_frame_frac * len(ida)))
                for dd, i, j in sorted((v[0], i, v[1]) for i, v in best.items())[:frame_cap]:
                    div_cands.append((dd, int(ida[i]), int(idb[j])))

    if divisions and div_cands:                                      # keep the globally most convincing ones
        for dd, u, v in sorted(div_cands)[:max(1, int(div_global_frac * N))]:
            edges.append((u, v))
            parent.setdefault(v, u)

    extra_rows = []
    if gap_close and len(by_t) > 2:
        out_deg = Counter(u for u, _ in edges)
        in_deg = Counter(v for _, v in edges)
        next_id, cap, added = int(nodes["node_id"].max()) + 1, int(gap_max_frac * N), 0
        for t in sorted(by_t):
            if t + 2 not in by_t or added >= cap:
                continue
            ida, pa = by_t[t]
            idc, pc = by_t[t + 2]
            A = [i for i, n in enumerate(ida) if out_deg[int(n)] == 0]
            C = [j for j, n in enumerate(idc) if in_deg[int(n)] == 0]
            if not A or not C:
                continue
            kk = min(3, len(C))
            d, k = cKDTree(pc[C]).query(pa[A], k=kk, distance_upper_bound=gap_um)
            d, k = d.reshape(len(A), kk), k.reshape(len(A), kk)
            cands = sorted((d[q, w], A[q], C[k[q, w]]) for q in range(len(A)) for w in range(kk) if np.isfinite(d[q, w]))
            used_a, used_c = set(), set()
            for dd, i, j in cands:
                if i in used_a or j in used_c or added >= cap:
                    continue
                used_a.add(i); used_c.add(j)
                mid = np.rint((pa[i] + pc[j]) / 2 / vs).astype(np.int64)
                extra_rows.append((next_id, t + 1, int(mid[0]), int(mid[1]), int(mid[2])))
                edges += [(int(ida[i]), next_id), (next_id, int(idc[j]))]
                next_id += 1; added += 1

    nodes_out = pd.concat([nodes, pd.DataFrame(extra_rows, columns=nodes.columns, dtype="int64")], ignore_index=True)
    edges_df = pd.DataFrame(edges, columns=["source_id", "target_id"], dtype="int64").drop_duplicates().reset_index(drop=True)
    return nodes_out, edges_df


In [11]:
def local_edge_metrics(pred_nodes, pred_edges, gt_nodes, gt_edges, scale, n_frames=None, tol_um=7.0):
    """Local version of the competition's edge Jaccard (rules from metrics.md): optimal 7 um node matching per frame,
    TP = predicted edge between matched nodes that is a GT edge, FP = predicted edge that contradicts a GT edge,
    FN = GT edge not recovered. Unmatched predicted nodes are NOT penalised. Does not include the node-count factor."""
    vs = np.asarray(scale, dtype=float)
    zyx = ["z", "y", "x"]
    if n_frames is not None:
        gt_nodes, pred_nodes = gt_nodes[gt_nodes.t < n_frames], pred_nodes[pred_nodes.t < n_frames]
    gt_ids = set(gt_nodes.node_id.astype(int))
    gt_e = gt_edges[gt_edges.source_id.isin(gt_ids) & gt_edges.target_id.isin(gt_ids)]
    gt_in, gt_out = defaultdict(set), defaultdict(set)
    for u, v in zip(gt_e.source_id.astype(int), gt_e.target_id.astype(int)):
        gt_out[u].add(v); gt_in[v].add(u)
    gt_edge_set = {(u, v) for u, vs_ in gt_out.items() for v in vs_}
    p2g = {}
    for t, g in gt_nodes.groupby("t"):
        p = pred_nodes[pred_nodes.t == t]
        if p.empty:
            continue
        cost = cdist(p[zyx].to_numpy() * vs, g[zyx].to_numpy() * vs)
        cost = np.where(cost <= tol_um, cost, 1e6)
        r, c = linear_sum_assignment(cost)
        pid, gid = p["node_id"].to_numpy(), g["node_id"].to_numpy()
        for i, j in zip(r, c):
            if cost[i, j] < 1e6:
                p2g[int(pid[i])] = int(gid[j])
    keep = set(pred_nodes.node_id.astype(int))
    pe = pred_edges[pred_edges.source_id.isin(keep) & pred_edges.target_id.isin(keep)]
    tp = fp = 0
    hit = set()
    for u, v in zip(pe.source_id.astype(int), pe.target_id.astype(int)):
        gu, gv = p2g.get(u), p2g.get(v)
        if gu is not None and gv is not None and (gu, gv) in gt_edge_set:
            tp += 1; hit.add((gu, gv))
            continue
        if (gv is not None and any(s != gu for s in gt_in.get(gv, ()))) or \
           (gu is not None and any(tg != gv for tg in gt_out.get(gu, ()))):
            fp += 1
    fn = len(gt_edge_set) - len(hit)
    return {"nodes/frame": round(len(pred_nodes) / max(pred_nodes.t.nunique(), 1), 1),
            "node_recall": round(len(p2g) / max(len(gt_nodes), 1), 3),
            "TP": tp, "FP": fp, "FN": fn, "edge_jaccard": round(tp / max(tp + fp + fn, 1), 4)}


In [12]:
def run_pipeline_selftest():
    """No data needed: synthetic cells drift, some drop out for a frame, one divides. Checks linking and the local metric."""
    rng = np.random.default_rng(0)
    scale = (1.625, 0.40625, 0.40625)
    rows, gt_edges, nid, last = [], [], 1, {}
    for k in range(40):
        p0 = rng.uniform([5, 40, 40], [50, 400, 400]); v = rng.normal(0, 1, 3) * [0.2, 1.2, 1.2]
        for t in range(10):
            p = p0 + v * t
            if k < 5 and t == 4:
                continue                                           # 1-frame dropout: gap closing must bridge it
            rows.append((nid, t, *np.rint(p).astype(int)))
            if k in last and last[k][1] == t - 1:
                gt_edges.append((last[k][0], nid))
            last[k] = (nid, t); nid += 1
    gt = pd.DataFrame(rows, columns=["node_id", "t", "z", "y", "x"])
    gte = pd.DataFrame(gt_edges, columns=["source_id", "target_id"])
    nodes_out, edges = link_tracks(gt, scale, max_link_um=10.0, divisions=False)
    m = local_edge_metrics(nodes_out, edges, gt, gte, scale)
    print("perfect detections ->", m)
    assert m["node_recall"] == 1.0 and m["edge_jaccard"] > 0.95, m
    assert len(nodes_out) > len(gt), "gap closing should have added synthetic nodes"
    # a division: one parent, two daughters 3 um apart in the next frame
    d = pd.DataFrame([(1, 0, 10, 100, 100), (2, 1, 10, 100, 104), (3, 1, 10, 100, 96)], columns=["node_id", "t", "z", "y", "x"])
    _, e2 = link_tracks(d, scale, divisions=True, gap_close=False)
    assert sorted(map(tuple, e2.values.tolist())) == [(1, 2), (1, 3)], e2
    # sanity: edges only between consecutive frames
    t_of = nodes_out.set_index("node_id")["t"]
    assert ((t_of.loc[edges.target_id].values - t_of.loc[edges.source_id].values) == 1).all()
    print("pipeline self-test OK")

run_pipeline_selftest()


perfect detections -> {'nodes/frame': 40.0, 'node_recall': 1.0, 'TP': 350, 'FP': 0, 'FN': 0, 'edge_jaccard': 1.0}
pipeline self-test OK


## 7. Measure on held-out training movies (read this table before submitting)

For each held-out movie and each detector mode, this prints:
* `nodes/frame`: should be **in the hundreds**, not ~13 like the first version,
* `node_recall`: share of annotated cells found within 7 um (**the number to push up first**),
* `edge_jaccard`: local version of the official edge score (without the small node-count factor).

Tuning: if `node_recall` is low, lower `K_MAD` (and `SWIN_THRESHOLD`). If `FP` is large compared with `TP`, raise `MAX_LINK_UM` less or lower it. Re-run this cell only; it does not retrain.

In [13]:
modes = ["classical"] + (["swin", "hybrid"] if SWIN_READY else [])
report = []
for s in holdout_stems:
    img, scale = open_movie(TRAIN_DIR, s)
    gt_nodes, gt_edges = load_geff_graph(TRAIN_DIR / f"{s}.geff")
    for mode in modes:
        t0 = time.time()
        det = detect_movie(FirstFrames(img, CHECK_FRAMES), scale, mode)
        nodes_out, edges = link_tracks(det, scale)
        res = local_edge_metrics(nodes_out, edges, gt_nodes, gt_edges, scale, n_frames=CHECK_FRAMES)
        report.append({"movie": s, "mode": mode, **res, "sec": round(time.time() - t0)})
report_df = pd.DataFrame(report)
print(report_df.to_string(index=False) if len(report_df) else "no held-out movies available (training folder not found)")


        movie      mode  nodes/frame  node_recall  TP  FP  FN  edge_jaccard  sec
6bba_fc83837d classical        147.9        0.804  34   9  15        0.5862   10
6bba_fe670320 classical         54.8        1.000  50   7   5        0.8065   10


### 7b. Pick `K_MAD` and `MAX_LINK_UM` automatically (blob detector, held-out movies)

Tries a small grid and keeps the setting with the best pooled local edge Jaccard. If several settings are within 0.01 of the best, it takes the one with **fewer nodes**, because the official score has a small penalty for predicting many more nodes than there are cells. It only runs when `AUTO_TUNE = True` and held-out movies exist. The held-out sample is small (a few dozen true edges), so treat the choice as a good starting point, not as exact.

In [14]:
if AUTO_TUNE and holdout_stems:
    cache = {}
    for s in holdout_stems:
        img, scale = open_movie(TRAIN_DIR, s)
        gn, ge = load_geff_graph(TRAIN_DIR / f"{s}.geff")
        cache[s] = (FirstFrames(img, CHECK_FRAMES), scale, gn, ge)
    rows = []
    for k in [3.0, 3.5, 4.0, 5.0]:
        dets = {s: classical_detect_movie(v[0], v[1], k_mad=k, verbose=False) for s, v in cache.items()}
        for lk in [6.0, 8.0, 10.0, 14.0]:
            res = []
            for s, v in cache.items():
                n_out, e = link_tracks(dets[s], v[1], max_link_um=lk)
                res.append(local_edge_metrics(n_out, e, v[2], v[3], v[1], n_frames=CHECK_FRAMES))
            tp, fp, fn = (sum(r[q] for r in res) for q in ("TP", "FP", "FN"))
            rows.append({"K_MAD": k, "MAX_LINK_UM": lk, "nodes/frame": round(np.mean([r["nodes/frame"] for r in res]), 1),
                         "node_recall": round(np.mean([r["node_recall"] for r in res]), 3),
                         "TP": tp, "FP": fp, "FN": fn, "edge_jaccard": round(tp / max(tp + fp + fn, 1), 4)})
    sweep = pd.DataFrame(rows).sort_values("edge_jaccard", ascending=False).reset_index(drop=True)
    print(sweep.head(8).to_string(index=False))
    near = sweep[sweep.edge_jaccard >= sweep.edge_jaccard.iloc[0] - 0.01].sort_values("nodes/frame")
    K_MAD, MAX_LINK_UM = float(near.iloc[0].K_MAD), float(near.iloc[0].MAX_LINK_UM)
    print(f"\n-> using K_MAD={K_MAD}, MAX_LINK_UM={MAX_LINK_UM} for the test movies")
else:
    print("AUTO_TUNE off or no held-out movies -> keeping K_MAD =", K_MAD, "and MAX_LINK_UM =", MAX_LINK_UM)


 K_MAD  MAX_LINK_UM  nodes/frame  node_recall  TP  FP  FN  edge_jaccard
   3.0         10.0        104.6        0.902  84  16  20        0.7000
   3.5         10.0        102.5        0.902  84  16  20        0.7000
   5.0         10.0         99.5        0.902  84  16  20        0.7000
   4.0         10.0        101.4        0.902  84  16  20        0.7000
   4.0          6.0        103.0        0.910  81  14  23        0.6864
   5.0          6.0        101.2        0.910  81  14  23        0.6864
   3.5          6.0        104.2        0.910  81  14  23        0.6864
   3.0          6.0        106.1        0.910  81  14  23        0.6864

-> using K_MAD=5.0, MAX_LINK_UM=10.0 for the test movies


## 8. Predict the test movies and write `submission.csv`

Columns follow the sample submission; unused cells are `-1`. If one movie fails with the chosen mode it is redone with the blob detector, so the file is always written.

In [15]:
def build_submission_rows(dataset, nodes_df, edges_df):
    cols = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
    n = pd.DataFrame({"dataset": dataset, "row_type": "node", "node_id": nodes_df["node_id"].to_numpy(),
                      "t": nodes_df["t"].to_numpy(), "z": nodes_df["z"].to_numpy(),
                      "y": nodes_df["y"].to_numpy(), "x": nodes_df["x"].to_numpy()})
    e = pd.DataFrame({"dataset": dataset, "row_type": "edge",
                      "source_id": edges_df["source_id"].to_numpy(), "target_id": edges_df["target_id"].to_numpy()})
    out = pd.concat([n, e], ignore_index=True)
    for c in ["node_id", "t", "z", "y", "x", "source_id", "target_id"]:
        out[c] = out[c].fillna(-1).astype("int64")                  # sample_submission uses -1, not blanks
    out.insert(0, "id", 0)
    return out.reindex(columns=cols)


def write_submission(parts, path):
    sub = pd.concat(parts, ignore_index=True)
    sub["id"] = np.arange(len(sub))
    sub.to_csv(path, index=False)
    return sub


def predict_movie(stem):
    img, scale = open_movie(TEST_DIR, stem)
    try:
        det = detect_movie(img, scale, DETECTOR_MODE)
        nodes_out, edges = link_tracks(det, scale)
    except Exception:
        traceback.print_exc()
        print(f"   {stem}: failed with mode {DETECTOR_MODE} -> redoing with the blob detector")
        det = detect_movie(img, scale, "classical")
        nodes_out, edges = link_tracks(det, scale)
    return nodes_out, edges


test_stems = list_stems(TEST_DIR)
assert test_stems, f"No .zarr movies found in {TEST_DIR}"
parts = []
for s in test_stems:
    t0 = time.time()
    nodes_out, edges = predict_movie(s)
    n_frames = max(int(nodes_out.t.max()) + 1, 1)
    forks = int((edges.source_id.value_counts() > 1).sum())
    print(f"{s}: {len(nodes_out):,} nodes ({len(nodes_out) / n_frames:.0f}/frame), {len(edges):,} edges, {forks} divisions, {(time.time() - t0) / 60:.1f} min")
    parts.append(build_submission_rows(s, nodes_out, edges))

SUBMISSION_PATH = WORKING_DIR / "submission.csv"
sub = write_submission(parts, SUBMISSION_PATH)
print("wrote", SUBMISSION_PATH, "| rows:", f"{len(sub):,}", "| nodes:", f"{(sub.row_type == 'node').sum():,}", "| edges:", f"{(sub.row_type == 'edge').sum():,}")

if SAMPLE_SUBMISSION.exists():
    sample = pd.read_csv(SAMPLE_SUBMISSION)
    print("columns match sample :", list(sample.columns) == list(sub.columns))
    print("dtypes match sample  :", bool((sample.dtypes.astype(str) == sub.dtypes.astype(str)).all()))
    print("datasets match sample:", set(sample.dataset) == set(sub.dataset))
sub.head()


   (Swin not ready -> blob detector only)
   blob detector: frame 25/100
   blob detector: frame 50/100
   blob detector: frame 75/100
   blob detector: frame 100/100
44b6_0113de3b: 24,206 nodes (242/frame), 23,115 edges, 10 divisions, 1.9 min
   (Swin not ready -> blob detector only)
   blob detector: frame 25/100
   blob detector: frame 50/100
   blob detector: frame 75/100
   blob detector: frame 100/100
44b6_0b24845f: 6,171 nodes (62/frame), 4,858 edges, 7 divisions, 2.1 min
   (Swin not ready -> blob detector only)
   blob detector: frame 25/100
   blob detector: frame 50/100
   blob detector: frame 75/100
   blob detector: frame 100/100
6bba_05b6850b: 5,941 nodes (59/frame), 5,554 edges, 4 divisions, 1.9 min
   (Swin not ready -> blob detector only)
   blob detector: frame 25/100
   blob detector: frame 50/100
   blob detector: frame 75/100
   blob detector: frame 100/100
6bba_05db0fb1: 23,158 nodes (232/frame), 20,564 edges, 36 divisions, 2.1 min
wrote /kaggle/working/submission

,id,dataset,row_type,node_id,t,z,y,x,source_id,target_id
0,0,44b6_0113de3b,node,1,0,0,5,52,-1,-1
1,1,44b6_0113de3b,node,2,0,0,6,72,-1,-1
2,2,44b6_0113de3b,node,3,0,0,27,66,-1,-1
3,3,44b6_0113de3b,node,4,0,0,63,58,-1,-1
4,4,44b6_0113de3b,node,5,0,0,99,35,-1,-1
